Given a GitHub issue description, retrieve the source files most likely to require modification.

This task mirrors how AI coding agents work in practice: a developer reports a bug or feature request in natural language, and the agent must identify which files to read and edit.

**Corpus:** [SWE-bench Lite](https://www.swebench.com) — 300 real GitHub issues from popular Python repositories (astropy, django, matplotlib, scikit-learn, etc.), with gold-patch file paths as ground truth.

**Challenge:** Issue descriptions use natural language (error messages, expected behaviour) while documents are file paths (`django/db/models/sql/compiler.py`). Keyword adapters must handle this semantic gap; dense embeddings bridge it via learned representations.

In [ ]:
import contextlib, json, pathlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BENCHMARK = 'code-finding'
ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists():
        ROOT = _p; break
RESULTS_DIR = ROOT / 'results'

ADAPTERS = ['sqlite', 'lancedb', 'chromadb', 'tantivy', 'qdrant']
ADAPTER_LABELS = {
    'sqlite': 'SQLite FTS5', 'lancedb': 'LanceDB',
    'chromadb': 'ChromaDB', 'tantivy': 'Tantivy', 'qdrant': 'Qdrant'
}
_OUTER_BG = '#f5f3ef'; _PLOT_BG = '#ffffff'; _MUTED = '#6b6b6b'
_SPINE = '#d8d5d0'; _GRID = '#ebebeb'; _LABEL_CLR = '#7a7370'; _INK = '#1a1917'
_ADAPTER_COLORS = {'sqlite': '#bdb9b5', 'lancedb': '#3d8c7a', 'chromadb': '#4b7ebb', 'tantivy': '#d4952a', 'qdrant': '#edc948'}
_FALLBACK = ['#c96442', '#4b7ebb', '#3d8c7a', '#d4952a', '#bdb9b5']
_P50 = '#e8903a'; _P95 = '#7eb8d4'
_TS = 9; _LS = 8; _TIS = 11.5
mpl.rcParams.update({'figure.dpi': 96, 'font.family': 'sans-serif', 'font.size': _TS,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': _GRID, 'grid.linewidth': 0.7, 'grid.linestyle': '-', 'axes.axisbelow': True})

def _sty(fig, ax):
    fig.patch.set_facecolor(_OUTER_BG); ax.set_facecolor(_PLOT_BG)
    for s in ['left','bottom']: ax.spines[s].set_color(_SPINE); ax.spines[s].set_linewidth(0.7)
    ax.tick_params(axis='both', colors=_MUTED, labelsize=_TS, length=3, width=0.7)
    ax.xaxis.label.set_color(_MUTED); ax.yaxis.label.set_color(_MUTED)

def _bc(s, i): return _ADAPTER_COLORS.get(s.lower(), _FALLBACK[i % len(_FALLBACK)])

rows = []
for f in RESULTS_DIR.glob('**/*.json'):
    with contextlib.suppress(Exception): rows.append(json.loads(f.read_text()))
df = pd.DataFrame(rows) if rows else pd.DataFrame()
bdf = (df[df['benchmark'] == BENCHMARK]
       .sort_values('ndcg_at_10', ascending=False)
       .groupby('store').first()
       .reindex(ADAPTERS))
print(f'Results for {BENCHMARK}: {len(bdf.dropna(subset=["ndcg_at_10"])) if not bdf.empty else 0} adapters')

## Data and Search Overview

### Task Flow

```mermaid
flowchart LR
    I["GitHub issue\n(natural language\nbug report / feature)"] --> R["Retrieval\n(query corpus of\nsource files)"]
    R --> F["Ranked source files\n(paths, no content)"]
    F --> P{"Correct file\nin top-k?"}
    P -->|Yes| Hit["Relevant hit\n→ nDCG gain"]
    P -->|No| Miss["Miss\n→ zero gain"]
    style I fill:#e3f2fd,stroke:#1565c0
    style Hit fill:#e8f5e9,stroke:#2e7d32
    style Miss fill:#fce4ec,stroke:#c62828
```


In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists(): ROOT = _p; break

result_dir = ROOT / 'results' / 'code-finding' / 'lancedb'
files = sorted(result_dir.glob('*.json')) if result_dir.exists() else []
meta = json.loads(files[-1].read_text()) if files else {}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
fig.patch.set_facecolor('#fafafa')
for ax in axes:
    ax.set_facecolor('#fafafa')
    for s in ax.spines.values(): s.set_visible(False)

# Left: corpus and query count
labels = ['Source files\n(documents)', 'GitHub issues\n(queries)']
vals = [meta.get('num_docs', 2294), meta.get('num_queries', 300)]
ax = axes[0]
bars = ax.bar(labels, vals, color=['#4e79a7', '#e15759'], width=0.5, zorder=3)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 30, f'{v:,}', ha='center', fontsize=10)
ax.set_title('Corpus scale', fontsize=10); ax.yaxis.grid(True, linestyle=':', alpha=0.6)

# Right: rank discount curve (DCG weight at each position)
ax2 = axes[1]
ranks = np.arange(1, 11)
weights = 1 / np.log2(ranks + 1)
ax2.bar(ranks, weights, color='#76b7b2', width=0.6, zorder=3)
ax2.set_xlabel('Rank position k', fontsize=9); ax2.set_ylabel('DCG weight 1/log2(k+1)', fontsize=9)
ax2.set_title('Position discount: rank 1 is worth 3x rank 5', fontsize=9)
ax2.set_xticks(ranks); ax2.yaxis.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout(); plt.show()
print(f"Corpus: {vals[0]:,} source files \u00b7 {vals[1]:,} issue queries")


## Background

### What This Benchmark Measures

Given a GitHub issue (bug report or feature request written in natural language), retrieve the source files most likely to need modification. This mirrors how AI coding agents work: a developer files an issue and the agent must identify which files to read and edit before writing a patch.

**Ground truth:** Derived from SWE-bench Lite — 300 real GitHub issues from 11 Python repositories, each paired with the set of files actually modified in the accepted fix. A retrieval is correct if the modified file appears in the top-k results.

**Why this is hard:** Issue vocabulary (human language describing symptoms) diverges sharply from file vocabulary (code identifiers, module names). A bug report saying "the login flow ignores the remember-me checkbox" must match `auth/session.py` or `views/account.py` — none of those tokens appear in the issue description.

### Why Dense Search Leads

Dense vector search projects both the issue description and file paths/content into the same semantic space. The encoder (all-MiniLM-L6-v2) has learned that "authentication failure" is semantically close to `auth/middleware.py` even without shared terms.

BM25 can only match on shared tokens. File paths and short module names have low term overlap with natural-language issue descriptions, leaving BM25 at a fundamental disadvantage on this task.

### References

1. Jimenez, C. E. et al. (2024). *SWE-bench: Can language models resolve real-world GitHub issues?* ICLR 2024. [arXiv:2310.06770](https://arxiv.org/abs/2310.06770)
2. [SWE-bench Lite dataset on HuggingFace](https://huggingface.co/datasets/princeton-nlp/SWE-bench_Lite)
3. Reimers, N. & Gurevych, I. (2019). Sentence-BERT. EMNLP 2019. [arXiv:1908.10084](https://arxiv.org/abs/1908.10084)
4. [LanceDB HNSW index documentation](https://lancedb.github.io/lancedb/concepts/index_ivfpq/)
5. [ChromaDB: getting started](https://docs.trychroma.com/getting-started)


## Results

In [ ]:
# Results table
cols = ['Adapter', 'nDCG@10', 'R@1', 'R@5', 'R@10', 'MRR@10', 'p50 (ms)']
rows_t = []
for a in ADAPTERS:
    if bdf.empty or a not in bdf.index or pd.isna(bdf.loc[a].get('ndcg_at_10')): continue
    r = bdf.loc[a]
    rows_t.append({'Adapter': ADAPTER_LABELS[a], 'nDCG@10': f"{r.get('ndcg_at_10',0):.3f}",
        'R@1': f"{r.get('recall_at_1',0):.3f}", 'R@5': f"{r.get('recall_at_5',0):.3f}",
        'R@10': f"{r.get('recall_at_10',0):.3f}", 'MRR@10': f"{r.get('mrr_at_10',0):.3f}",
        'p50 (ms)': f"{r.get('latency_p50_ms',0):.2f}"})
if rows_t:
    from IPython.display import display, HTML
    tdf = pd.DataFrame(rows_t, columns=cols)
    display(HTML(tdf.to_html(index=False, classes='results-table', border=0)))
else:
    from IPython.display import display, HTML
    display(HTML('<p><em>No results.</em></p>'))

In [ ]:
# nDCG@10 by adapter
valid = [(a, bdf.loc[a,'ndcg_at_10']) for a in ADAPTERS if not bdf.empty and a in bdf.index and not pd.isna(bdf.loc[a,'ndcg_at_10'])]
if valid:
    stores, vals = zip(*valid)
    labels = [ADAPTER_LABELS.get(s,s) for s in stores]
    colors = [_bc(s,i) for i,s in enumerate(stores)]
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    _sty(fig, ax)
    bars = ax.bar(labels, vals, color=colors, width=0.5, zorder=3)
    ax.set_ylabel('nDCG@10'); ax.set_ylim(0, 1.1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015, f'{val:.3f}',
                ha='center', va='bottom', fontsize=_LS, color=_LABEL_CLR)
    ax.set_title('nDCG@10 by adapter', color=_INK, fontsize=_TIS, fontweight='bold', pad=10)
    plt.tight_layout(pad=1.0); plt.show()

In [ ]:
# Query latency (p50 and p95)
valid_lat = [(a, bdf.loc[a,'latency_p50_ms'], bdf.loc[a,'latency_p95_ms'])
             for a in ADAPTERS if not bdf.empty and a in bdf.index and not pd.isna(bdf.loc[a].get('latency_p50_ms'))]
fig, ax = plt.subplots(figsize=(5.5, 3.2))
_sty(fig, ax)
if valid_lat:
    stores, p50, p95 = zip(*valid_lat)
    labels = [ADAPTER_LABELS.get(s,s) for s in stores]
    x = np.arange(len(labels)); w = 0.3
    ax.bar(x-w/2, p50, w, label='p50', color=_P50, zorder=3)
    ax.bar(x+w/2, p95, w, label='p95', color=_P95, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('ms')
    ax.legend(fontsize=_LS, framealpha=0, labelcolor=_MUTED, handlelength=1.0)
else:
    ax.text(0.5, 0.5, 'No latency data', ha='center', va='center', color=_MUTED, transform=ax.transAxes)
ax.set_title('Query latency (ms)', color=_INK, fontsize=_TIS, fontweight='bold', pad=10)
plt.tight_layout(pad=1.0); plt.show()

## Analysis

LanceDB leads at nDCG@10=0.623, followed closely by ChromaDB (0.585). Both dense-vector adapters substantially outperform keyword search, confirming that semantic similarity is essential when the query vocabulary (issue descriptions) differs from the document vocabulary (file paths).

Tantivy (0.385) and SQLite FTS5 (0.394) perform similarly despite their architectural differences, suggesting that BM25 keyword matching is equally limited when there is little term overlap. The small gap between BM25 adapters and dense adapters may close if file contents rather than just file paths are indexed.

## Limitations

- **File paths only:** documents are file path strings, not file contents. Indexing actual code would dramatically improve all adapters.
- **No cross-file reasoning:** some bugs require changes across multiple files; single-document retrieval cannot model this.
- **Oracle labels:** ground truth is the gold-patch diff, which may not include all files a developer would want to read.